# MAJ-Debate: Stage 3 Argumentation Framework Engine

This notebook consumes split-specific Stage 2 outputs, computes Dung-style semantics, and writes split-specific Stage 3 outputs.


## 0. Imports & Configuration


In [ ]:
import os, json, statistics
from pathlib import Path
from datetime import datetime
from itertools import chain, combinations

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://YOUR_VM_IP:5000')
TOPIC_LIMIT = int(os.environ.get('MAJ_STAGE3_TOPIC_LIMIT', '0'))
INCLUDE_SUPPORT = os.environ.get('MAJ_STAGE3_INCLUDE_SUPPORT', '1') == '1'
COMPUTE_PREFERRED_ONLY_IF_NEEDED = os.environ.get('MAJ_STAGE3_PREFERRED_ONLY_IF_NEEDED', '1') == '1'
EVAL_SPLIT = os.environ.get('MAJ_EVAL_SPLIT', 'ddo_sample')

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists()), cwd)
IN_FILE = PROJECT_ROOT / 'outputs' / 'stage2' / EVAL_SPLIT / 'stage2_relations.json'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'stage3' / EVAL_SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'stage3_graphs.json'
FAILURES_FILE = OUT_DIR / 'stage3_failures.json'
RESUME_EXISTING = os.environ.get('MAJ_STAGE3_RESUME', '1').strip() not in {'0', 'false', 'False'}
CONTINUE_ON_ERROR = os.environ.get('MAJ_STAGE3_CONTINUE_ON_ERROR', '1').strip() not in {'0', 'false', 'False'}
print(f'Input   : {IN_FILE.resolve()}')
print(f'Output  : {OUT_FILE.resolve()}')
print(f'Failures: {FAILURES_FILE.resolve()}')
print(f'Resume  : {RESUME_EXISTING}')
print(f'Continue: {CONTINUE_ON_ERROR}')
print(f'Preferred only if needed: {COMPUTE_PREFERRED_ONLY_IF_NEEDED}')


## 1. Dung Semantics Helpers


In [ ]:
def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s) + 1))

def attackers_of(att_edges, x):
    return {src for src, tgt in att_edges if tgt == x}

def conflict_free(candidate, att_edges):
    cand = set(candidate)
    return all((a, b) not in att_edges for a in cand for b in cand if a != b)

def defended_by(candidate, x, att_edges):
    cand = set(candidate)
    atkers = attackers_of(att_edges, x)
    if not atkers:
        return True
    return all(any((c, y) in att_edges for c in cand) for y in atkers)

def admissible(candidate, att_edges):
    cand = set(candidate)
    return conflict_free(cand, att_edges) and all(defended_by(cand, x, att_edges) for x in cand)

def characteristic(att_edges, arguments, ext):
    return {x for x in arguments if defended_by(ext, x, att_edges)}

def grounded_extension(arguments, att_edges):
    current = set()
    while True:
        nxt = characteristic(att_edges, arguments, current)
        if nxt == current:
            return current
        current = nxt

def preferred_extensions(arguments, att_edges):
    admissible_sets = [set(p) for p in powerset(arguments) if admissible(set(p), att_edges)]
    maximal = []
    for cand in admissible_sets:
        if not any(cand < other for other in admissible_sets):
            maximal.append(cand)
    unique = []
    seen = set()
    for ext in maximal:
        key = tuple(sorted(ext))
        if key not in seen:
            seen.add(key)
            unique.append(sorted(ext))
    return unique

def side_vote(extension, arg_index):
    counts = {'PRO': 0, 'CON': 0}
    for arg_id in extension:
        stance = arg_index[arg_id]['stance']
        if stance in counts:
            counts[stance] += 1
    if counts['PRO'] > counts['CON']:
        return 'PRO'
    if counts['CON'] > counts['PRO']:
        return 'CON'
    return 'UNDECIDED'


## 2. Core Graph Functions


In [ ]:
def load_stage2(path=IN_FILE):
    if not path.exists():
        raise FileNotFoundError(f'Stage 2 input not found: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def relation_justification(rel):
    text = rel.get('justification')
    if isinstance(text, str) and text.strip():
        return text.strip()
    label = rel.get('label', 'relation')
    conf = rel.get('confidence')
    premise = rel.get('attacked_premise')
    if premise:
        return f"{label} relation inferred from Stage 2 output; attacked premise: {premise}"
    if isinstance(conf, (int, float)):
        return f"{label} relation inferred from Stage 2 output with confidence {conf:.3f}."
    return f"{label} relation inferred from Stage 2 output."

def build_topic_graph(topic_rec, include_support=True):
    args = topic_rec['arguments']
    arg_index = {a['arg_id']: a for a in args}
    attack_edges = []
    support_edges = []
    for rel in topic_rec['relations']:
        if not rel.get('kept'):
            continue
        edge = {'source_arg_id': rel['source_arg_id'], 'target_arg_id': rel['target_arg_id'], 'confidence': rel.get('confidence', 0.0), 'justification': relation_justification(rel), 'attacked_premise': rel.get('attacked_premise')}
        if rel['label'] == 'Attack':
            attack_edges.append(edge)
        elif include_support and rel['label'] == 'Support':
            support_edges.append(edge)
    att_edge_set = {(e['source_arg_id'], e['target_arg_id']) for e in attack_edges}
    grounded = sorted(grounded_extension(arg_index.keys(), att_edge_set))
    grounded_vote = side_vote(grounded, arg_index)
    preferred = []
    fallback_reason = None
    preferred_vote = grounded_vote
    if grounded_vote == 'UNDECIDED':
        fallback_reason = 'grounded_winner_unresolved'
        preferred = preferred_extensions(arg_index.keys(), att_edge_set)
        pref_votes = [side_vote(ext, arg_index) for ext in preferred]
        if pref_votes.count('PRO') > pref_votes.count('CON'):
            preferred_vote = 'PRO'
        elif pref_votes.count('CON') > pref_votes.count('PRO'):
            preferred_vote = 'CON'
        else:
            preferred_vote = 'UNDECIDED'
    elif not COMPUTE_PREFERRED_ONLY_IF_NEEDED:
        preferred = preferred_extensions(arg_index.keys(), att_edge_set)
    chosen_extension = grounded if grounded_vote != 'UNDECIDED' else (preferred[0] if preferred else [])
    decisive_edges = [e for e in attack_edges if e['source_arg_id'] in chosen_extension and e['target_arg_id'] not in chosen_extension]
    winner = grounded_vote if grounded_vote != 'UNDECIDED' else preferred_vote
    return {'topic_id': topic_rec['topic_id'], 'topic_text': topic_rec['topic_text'], 'domain': topic_rec.get('domain'), 'benchmark_label': topic_rec.get('benchmark_label'), 'source_dataset': topic_rec.get('source_dataset'), 'source_ref': topic_rec.get('source_ref'), 'evaluation_split': topic_rec.get('evaluation_split', EVAL_SPLIT), 'run_name': topic_rec.get('run_name'), 'arguments': args, 'argument_strength': topic_rec.get('argument_strength', {}), 'attack_graph': attack_edges, 'support_graph': support_edges, 'grounded_extension': grounded, 'preferred_extensions': preferred, 'winner': winner, 'fallback_reason': fallback_reason, 'decisive_attacks': decisive_edges, 'summary': {'n_arguments': len(args), 'n_attack_edges': len(attack_edges), 'n_support_edges': len(support_edges), 'grounded_extension_size': len(grounded), 'n_preferred_extensions': len(preferred), 'winner': winner, 'graph_stability': round(len(grounded) / max(len(args), 1), 3), 'disconnected_or_sparse': len(attack_edges) < max(1, len(args) // 3)}}

try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('MAJ-Debate')
    MLFLOW_OK = True
except Exception as ex:
    MLFLOW_OK = False
    print(f'MLflow unavailable ({ex}) - results saved locally only')

def mlflow_log(run_name, params, metrics, artifact_paths):
    if not MLFLOW_OK:
        return
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for path in artifact_paths:
            mlflow.log_artifact(str(path), artifact_path=f'stage3/{EVAL_SPLIT}')


## 3. Main Run - Stage 3 on All Topics


In [ ]:
stage2_doc = load_stage2()
topics = stage2_doc['topics'][:TOPIC_LIMIT] if TOPIC_LIMIT > 0 else stage2_doc['topics']
run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f'stage3-{EVAL_SPLIT}-{run_ts}'

def compute_stage3_summary(topic_payloads):
    return {'total_topics': len(topic_payloads), 'total_attack_edges': sum(t['summary']['n_attack_edges'] for t in topic_payloads), 'total_support_edges': sum(t['summary']['n_support_edges'] for t in topic_payloads), 'avg_grounded_extension_size': round(statistics.mean(t['summary']['grounded_extension_size'] for t in topic_payloads), 3) if topic_payloads else 0.0, 'avg_graph_stability': round(statistics.mean(t['summary']['graph_stability'] for t in topic_payloads), 3) if topic_payloads else 0.0, 'fallback_count': sum(1 for t in topic_payloads if t.get('fallback_reason'))}

def save_stage3_state(topic_results_by_id, failures):
    ordered_topics = [topic_results_by_id[t['topic_id']] for t in topics if t['topic_id'] in topic_results_by_id]
    output_doc = {'stage': 3, 'run_name': run_name, 'timestamp': run_ts, 'config': {'evaluation_split': EVAL_SPLIT, 'include_support': INCLUDE_SUPPORT, 'preferred_only_if_needed': COMPUTE_PREFERRED_ONLY_IF_NEEDED, 'source_stage2_run': stage2_doc.get('run_name'), 'resume_existing': RESUME_EXISTING}, 'topics': ordered_topics, 'summary': compute_stage3_summary(ordered_topics)}
    with open(OUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(output_doc, f, indent=2)
    with open(FAILURES_FILE, 'w', encoding='utf-8') as f:
        json.dump({'run_name': run_name, 'timestamp': run_ts, 'failed_topics': failures}, f, indent=2)
    return output_doc

existing_topics_by_id = {}
if RESUME_EXISTING and OUT_FILE.exists():
    try:
        with open(OUT_FILE, 'r', encoding='utf-8') as f:
            existing_doc = json.load(f)
        for topic_payload in existing_doc.get('topics', []):
            topic_id = topic_payload.get('topic_id')
            if topic_id and topic_payload.get('summary'):
                existing_topics_by_id[topic_id] = topic_payload
        print(f'Resuming from existing output: {len(existing_topics_by_id)} completed topics found')
    except Exception as ex:
        print(f'Could not resume from existing output ({ex}); starting fresh')
        existing_topics_by_id = {}

existing_failures_by_id = {}
if FAILURES_FILE.exists():
    try:
        with open(FAILURES_FILE, 'r', encoding='utf-8') as f:
            existing_failures_doc = json.load(f)
        for failure_payload in existing_failures_doc.get('failed_topics', []):
            failed_topic_id = failure_payload.get('topic_id')
            if failed_topic_id and failed_topic_id not in existing_topics_by_id:
                existing_failures_by_id[failed_topic_id] = failure_payload
    except Exception as ex:
        print(f'Could not load existing failures ({ex}); continuing with a fresh failure log')
        existing_failures_by_id = {}

topic_results_by_id = dict(existing_topics_by_id)
failures_by_id = dict(existing_failures_by_id)
for idx, topic in enumerate(topics, start=1):
    topic_id = topic['topic_id']
    if topic_id in topic_results_by_id:
        print(f"[{idx}/{len(topics)}] {topic_id}: already completed, skipping")
        continue
    print(f"[{idx}/{len(topics)}] {topic_id}: {topic['topic_text']}")
    try:
        topic_results_by_id[topic_id] = build_topic_graph(topic, include_support=INCLUDE_SUPPORT)
        failures_by_id.pop(topic_id, None)
        output_doc = save_stage3_state(topic_results_by_id, list(failures_by_id.values()))
        print(f"  saved checkpoint after {topic_id}")
    except Exception as ex:
        failures_by_id[topic_id] = {'topic_id': topic_id, 'topic_text': topic['topic_text'], 'error': str(ex), 'run_name': run_name}
        output_doc = save_stage3_state(topic_results_by_id, list(failures_by_id.values()))
        if not CONTINUE_ON_ERROR:
            raise

output_doc = save_stage3_state(topic_results_by_id, list(failures_by_id.values()))
mlflow_log(run_name=run_name, params=output_doc['config'], metrics={k: float(v) for k, v in output_doc['summary'].items() if isinstance(v, (int, float))}, artifact_paths=[OUT_FILE, FAILURES_FILE])
print(f'Saved: {OUT_FILE.resolve()}')
print(output_doc['summary'])
print(f'Failed topics: {len(failures_by_id)}')


## 4. Inspect Output


In [ ]:
sample = output_doc['topics'][0]
print('Topic    :', sample['topic_text'])
print('Winner   :', sample['winner'])
print('Label    :', sample['benchmark_label'])
print('Fallback :', sample['fallback_reason'])
